# Automated EDA & Complete Pipeline

Automated EDA tools generate a comprehensive HTML report from a single function call — useful for quickly profiling a new dataset before diving into manual analysis.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

df = sns.load_dataset('titanic')
print("Shape:", df.shape)

Shape: (891, 15)


---
## ydata-profiling (formerly pandas-profiling)

Generates a rich interactive report with distributions, correlations, missing value analysis, and duplicates.

```bash
pip install ydata-profiling
```

In [2]:
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(
        df,
        title='Titanic EDA Report',
        explorative=True,
        minimal=False
    )
    # Save to HTML
    profile.to_file('titanic_ydata_report.html')
    print("Report saved to titanic_ydata_report.html")

    # In Jupyter: display inline
    # profile.to_notebook_iframe()

except ImportError:
    print("ydata-profiling not installed. Run: pip install ydata-profiling")
    print("The code above shows the correct usage pattern.")

ydata-profiling not installed. Run: pip install ydata-profiling
The code above shows the correct usage pattern.


### What ydata-profiling reports

| Section | Contents |
|---|---|
| **Overview** | Dataset shape, type counts, duplicate count |
| **Variables** | Distribution, missing %, min/max/mean for each column |
| **Interactions** | Scatter plots for all pairs |
| **Correlations** | Pearson, Spearman, Cramér's V (for categorical) |
| **Missing values** | Heatmap and bar chart of missingness |
| **Sample** | First and last rows |

---
## sweetviz

Specialises in **comparison reports** — comparing train vs test sets, or two datasets directly.

```bash
pip install sweetviz
```

In [3]:
try:
    import sweetviz as sv
    from sklearn.model_selection import train_test_split

    train, test = train_test_split(df, test_size=0.2, random_state=42)

    # Compare train vs test
    report = sv.compare(
        [train, 'Train'],
        [test,  'Test'],
        target_feat='survived'
    )
    report.show_html('titanic_sweetviz_report.html', open_browser=False)
    print("Sweetviz report saved to titanic_sweetviz_report.html")

except ImportError:
    print("sweetviz not installed. Run: pip install sweetviz")
    print("The code above shows the correct usage pattern.")

sweetviz not installed. Run: pip install sweetviz
The code above shows the correct usage pattern.


### When to use each tool

| Tool | Best for |
|---|---|
| **ydata-profiling** | Deep exploration of a single dataset |
| **sweetviz** | Comparing train vs test; target-aware analysis |

---
## Complete EDA Pipeline

Putting all the concepts from notebooks 1–16 into a single, reusable pipeline:

In [4]:
def run_eda_pipeline(df_raw, target_col=None, drop_thresh=0.4):
    """Run a complete EDA pipeline and return a cleaned, encoded DataFrame."""
    print("=" * 60)
    print("STEP 1 — Dataset Overview")
    print("=" * 60)
    print(f"Shape: {df_raw.shape}")
    print(f"Columns: {df_raw.columns.tolist()}")

    df = df_raw.copy()

    print("\n" + "=" * 60)
    print("STEP 2 — Remove Duplicates")
    print("=" * 60)
    n_before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {n_before - len(df)} duplicate rows")

    print("\n" + "=" * 60)
    print("STEP 3 — Missing Values")
    print("=" * 60)
    missing = df.isnull().mean().round(3)
    drop_cols = missing[missing > drop_thresh].index.tolist()
    print(f"Columns with >{drop_thresh*100:.0f}% missing → dropped: {drop_cols}")
    df = df.drop(columns=drop_cols)

    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        if col == target_col:
            continue
        df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(include=['object', 'category']).columns:
        if col == target_col:
            continue
        df[col] = df[col].fillna(df[col].mode()[0])

    print(f"Remaining missing: {df.isnull().sum().sum()}")

    print("\n" + "=" * 60)
    print("STEP 4 — Outlier Capping (IQR × 1.5)")
    print("=" * 60)
    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        if col == target_col:
            continue
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower=lower, upper=upper)
        if n_out:
            print(f"  {col}: {n_out} values capped")

    print("\n" + "=" * 60)
    print("STEP 5 — Encoding Categorical Columns")
    print("=" * 60)
    cat_cols = [c for c in df.select_dtypes(include=['object', 'category']).columns if c != target_col]
    print(f"One-hot encoding: {cat_cols}")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    print("\n" + "=" * 60)
    print("STEP 6 — Feature Scaling")
    print("=" * 60)
    num_cols = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                if c != target_col]
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])
    print(f"Scaled {len(num_cols)} numerical columns")

    print("\n" + "=" * 60)
    print(f"FINAL SHAPE: {df.shape}")
    print("=" * 60)

    return df


df_final = run_eda_pipeline(df, target_col='survived')

STEP 1 — Dataset Overview
Shape: (891, 15)
Columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

STEP 2 — Remove Duplicates
Removed 107 duplicate rows

STEP 3 — Missing Values
Columns with >40% missing → dropped: ['deck']
Remaining missing: 0

STEP 4 — Outlier Capping (IQR × 1.5)
  age: 39 values capped
  sibsp: 39 values capped
  parch: 15 values capped
  fare: 102 values capped

STEP 5 — Encoding Categorical Columns
One-hot encoding: ['sex', 'embarked', 'class', 'who', 'embark_town', 'alive']

STEP 6 — Feature Scaling
Scaled 5 numerical columns

FINAL SHAPE: (784, 18)


In [5]:
print("Final feature set:")
print(df_final.dtypes)
print()
print("First 3 rows:")
print(df_final.head(3))

Final feature set:
survived                     int64
pclass                     float64
age                        float64
sibsp                      float64
parch                      float64
fare                       float64
adult_male                    bool
alone                         bool
sex_male                      bool
embarked_Q                    bool
embarked_S                    bool
class_Second                  bool
class_Third                   bool
who_man                       bool
who_woman                     bool
embark_town_Queenstown        bool
embark_town_Southampton       bool
alive_yes                     bool
dtype: object

First 3 rows:
   survived    pclass       age     sibsp     parch      fare  adult_male  \
0         0  0.885158 -0.565647  0.776125 -0.543995 -0.845924        True   
1         1 -1.455362  0.661944  0.776125 -0.543995  1.969018       False   
2         1  0.885158 -0.258749 -0.634030 -0.543995 -0.816251       False   

   alone  sex

---
## The EDA Checklist

Run through this checklist every time you work with a new dataset:

| Step | Action | Notebook |
|---|---|---|
| 1 | Understand the problem and data types | nb_01, nb_02 |
| 2 | Central tendency: mean, median, mode | nb_03 |
| 3 | Dispersion and skewness | nb_04 |
| 4 | Detect and handle missing values | nb_05 |
| 5 | Remove duplicates and cap outliers | nb_06 |
| 6 | Standardise strings, dates, units | nb_07 |
| 7 | Compute correlation matrix; check heatmap | nb_08 |
| 8 | Mutual information — non-linear feature ranking | nb_09 |
| 9 | Encode categorical columns | nb_10 |
| 10 | Scale numerical features | nb_11 |
| 11 | Univariate EDA: profile each variable | nb_12 |
| 12 | Run the 4-Plot to check EDA assumptions | nb_12 |
| 13 | Bivariate EDA: pairwise relationships + statistical tests | nb_13 |
| 14 | Multivariate EDA: patterns across 3+ variables | nb_14 |
| 15 | Engineer new features | nb_15 |
| 16 | Audit for data leakage | nb_16 |
| 17 | Generate automated report for quick summary | nb_17 |